# Preparation: DICOM → NumPy + metadata

DICOMs are slow to decode and will not be used for training. This notebook converts every series in `input/train_series/` once:

- **Pixels** → one 3D `.npy` per series, shape `(n_slices, rows, cols)`, in the **native dtype** (uint16 / int16) with **no rescaling, windowing or resampling**, so nothing is lost. Slices are ordered along the slice normal (position projected on the slice normal of the majority orientation; an `orientation` column is added).
- **Metadata** → `output/metadata.csv`, one row per slice (in stacked order), containing every DICOM tag except `PixelData`. `RescaleSlope/Intercept`, spacing and orientation are kept there, so the physical values can be recomputed from the raw arrays.
- **Series index** → `output/series_index.csv`, one row per series with the array path, shape and dtype.

Set `LIMIT` to a number to process only a random subset. `None` processes everything. The step is resumable: series that already have a shard are skipped.

In [ ]:
import json, os, time, warnings
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

ROOT = Path('..').resolve()
INPUT = ROOT / 'input'
OUT = ROOT / 'output'
ARR_DIR = OUT / 'series_npy'          # <SeriesInstanceUID>.npy
SHARD_DIR = OUT / 'metadata_shards'   # per-series metadata, makes the run resumable
for d in (ARR_DIR, SHARD_DIR):
    d.mkdir(parents=True, exist_ok=True)

LIMIT = None      # e.g. 300 for a quick test run, None for all series
N_WORKERS = 64
SEED = 0

## 1. The tabular data

In [ ]:
train = pd.read_csv(INPUT / 'train.csv')
series = pd.read_csv(INPUT / 'train_series.csv')
LABELS = [c for c in train.columns if c not in ('StudyInstanceUID', 'Report')]
print(f'{len(train):,} studies, {len(series):,} series, {series.groupby("StudyInstanceUID").size().describe()[["min","50%","max"]].tolist()} series per study (min/median/max)')
print(f'{train[LABELS[0]].notna().sum()} studies have labels; the rest only have a report.')
series.head()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
series['Anatomical_Plane'].value_counts().plot.bar(ax=ax[0], title='Series per plane', rot=0)
series.groupby(['Fluid_Sensitive', 'Fat_Suppression']).size().plot.bar(ax=ax[1], title='Series by (Fluid_Sensitive, Fat_Suppression)', rot=0)
series.groupby('StudyInstanceUID').size().value_counts().sort_index().plot.bar(ax=ax[2], title='Series per study', rot=0)
plt.tight_layout(); plt.show()

In [ ]:
lab = train.dropna(subset=LABELS)
print(f'Label prevalence over the {len(lab)} labeled studies')
lab[LABELS].mean().sort_values().plot.barh(figsize=(6, 4)); plt.xlabel('positive rate'); plt.show()

In [ ]:
pd.set_option('display.max_colwidth', 300)
train[['StudyInstanceUID', 'Report']].sample(5, random_state=1)

## 2. Conversion

For each series: read all slices, sort by position along the slice normal (tie-break `InstanceNumber`), stack the raw `pixel_array`s, save with `np.save` (uncompressed, so it can be loaded fast or memory-mapped), and keep all non-pixel tags.

Checks done per series: all slices have the same shape and dtype, otherwise the series is reported as failed instead of being silently altered.

In [ ]:
def _clean(v):
    """Turn a pydicom value into something CSV friendly without losing information."""
    if isinstance(v, pydicom.multival.MultiValue):
        return json.dumps([_clean(x) for x in v])
    if isinstance(v, (pydicom.valuerep.DSfloat, pydicom.valuerep.DSdecimal)):
        return float(v)
    if isinstance(v, pydicom.valuerep.IS):
        return int(v)
    if isinstance(v, (bytes, bytearray)):
        return v.hex()
    if isinstance(v, (str, int, float, bool)) or v is None:
        return v
    return str(v)


def slice_meta(ds):
    row = {}
    for e in ds:
        if e.tag == 0x7FE00010:   # PixelData
            continue
        key = e.keyword or f'Tag_{e.tag.group:04X}{e.tag.element:04X}'
        row[key] = f'<sequence of {len(e.value)}>' if e.VR == 'SQ' else _clean(e.value)
    row['TransferSyntax'] = ds.file_meta.TransferSyntaxUID.name
    return row


def get_orient(iop):
    """axial / sagittal / coronal from ImageOrientationPatient (dominant axis of the slice normal)."""
    if iop is None:
        return ''
    n = np.abs(np.cross(np.array(iop[:3], float), np.array(iop[3:], float)))
    return ['sagittal', 'coronal', 'axial'][int(np.argmax(n))]


def slice_order(dss):
    """Sort slices along the slice normal of the majority orientation.

    Returns (order, per-slice orientation, per-slice position along the normal). Slices whose own
    orientation is an outlier do not influence the normal; ties are broken with InstanceNumber.
    """
    inst = np.array([int(d.get('InstanceNumber', 0)) for d in dss])
    try:
        iops = [np.array(d.ImageOrientationPatient, float) for d in dss]
        orients = [get_orient(i) for i in iops]
        major = max(set(orients), key=orients.count)
        normals = np.array([np.cross(i[:3], i[3:]) for i in iops])
        normal = normals[[o == major for o in orients]].mean(0)
        normal /= np.linalg.norm(normal)
        pos = np.array([np.dot(np.array(d.ImagePositionPatient, float), normal) for d in dss])
        return np.lexsort((inst, pos)), [major] * len(dss), pos
    except Exception:
        return np.argsort(inst, kind='stable'), [''] * len(dss), inst.astype(float)


def convert_series(args):
    study_uid, series_uid = args
    arr_path, shard_path = ARR_DIR / f'{series_uid}.npy', SHARD_DIR / f'{series_uid}.pkl'
    if arr_path.exists() and shard_path.exists():
        return 'skipped', series_uid, None
    try:
        files = list((INPUT / 'train_series' / study_uid / series_uid).glob('*.dcm'))
        dss = [pydicom.dcmread(f) for f in files]
        order, orients, pos = slice_order(dss)
        dss = [dss[i] for i in order]
        arrs = [d.pixel_array for d in dss]
        arrs = [a[0] if a.ndim == 3 and a.shape[0] == 1 else a for a in arrs]   # single-frame multiframe
        if len({(a.shape, a.dtype) for a in arrs}) != 1:
            raise ValueError(f'inconsistent slices: {sorted({(a.shape, str(a.dtype)) for a in arrs})}')
        vol = np.stack(arrs)
        assert vol.ndim == 3
        np.save(arr_path.with_suffix('.tmp.npy'), vol)
        os.replace(arr_path.with_suffix('.tmp.npy'), arr_path)
        rows = []
        for i, d in enumerate(dss):
            r = slice_meta(d)
            r.update(slice_idx=i, n_slices=len(dss), orientation=orients[i], slice_position=float(pos[order[i]]), array_path=str(arr_path.relative_to(ROOT)))
            rows.append(r)
        pd.DataFrame(rows).to_pickle(shard_path)
        return 'ok', series_uid, None
    except Exception as ex:
        return 'failed', series_uid, repr(ex)

In [ ]:
todo = series if LIMIT is None else series.sample(LIMIT, random_state=SEED)
jobs = list(zip(todo.StudyInstanceUID, todo.SeriesInstanceUID))
t0 = time.time()
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    with ProcessPoolExecutor(N_WORKERS) as ex:
        results = list(tqdm(ex.map(convert_series, jobs, chunksize=4), total=len(jobs)))
status = pd.Series([r[0] for r in results]).value_counts().to_dict()
failed = [(r[1], r[2]) for r in results if r[0] == 'failed']
print(f'{status} in {time.time()-t0:.0f}s')
for f in failed[:20]:
    print('FAILED', f)

In [ ]:
# Assemble metadata.csv (one row per slice) and series_index.csv (one row per series)
shards = [SHARD_DIR / f'{s}.pkl' for s in todo.SeriesInstanceUID if (SHARD_DIR / f'{s}.pkl').exists()]
meta = pd.concat([pd.read_pickle(p) for p in shards], ignore_index=True)
front = ['StudyInstanceUID', 'SeriesInstanceUID', 'SOPInstanceUID', 'slice_idx', 'n_slices', 'orientation', 'slice_position', 'array_path']
meta = meta[front + [c for c in meta.columns if c not in front]]
meta = meta.merge(series[['SeriesInstanceUID', 'Fluid_Sensitive', 'Fat_Suppression', 'Anatomical_Plane']], on='SeriesInstanceUID', how='left')
meta.to_csv(OUT / 'metadata.csv', index=False)

idx = (meta.groupby('SeriesInstanceUID', sort=False)
           .agg(StudyInstanceUID=('StudyInstanceUID', 'first'), array_path=('array_path', 'first'),
                n_slices=('n_slices', 'first'), rows=('Rows', 'first'), cols=('Columns', 'first'),
                bits_stored=('BitsStored', 'first'), pixel_representation=('PixelRepresentation', 'first'))
           .reset_index())
idx['pixel_dtype'] = np.where(idx.pixel_representation == 1, 'int16', 'uint16')
idx = idx.merge(series[['SeriesInstanceUID', 'Fluid_Sensitive', 'Fat_Suppression', 'Anatomical_Plane']], on='SeriesInstanceUID')
idx.to_csv(OUT / 'series_index.csv', index=False)
print(meta.shape, idx.shape)

## 3. Verification

Reload a few arrays from disk and compare them bit-for-bit against a fresh DICOM decode.

In [ ]:
rng = np.random.default_rng(SEED)
for _, r in idx.sample(min(5, len(idx)), random_state=SEED).iterrows():
    vol = np.load(ROOT / r.array_path)
    m = meta[meta.SeriesInstanceUID == r.SeriesInstanceUID].sort_values('slice_idx')
    k = int(rng.integers(len(m)))
    ds = pydicom.dcmread(INPUT / 'train_series' / r.StudyInstanceUID / r.SeriesInstanceUID / f'{m.iloc[k].SOPInstanceUID}.dcm')
    assert vol.shape == (r.n_slices, r.rows, r.cols) and str(vol.dtype) == r.pixel_dtype
    assert np.array_equal(vol[k], ds.pixel_array), 'pixel mismatch'
    print(f'OK  {r.SeriesInstanceUID[-8:]}  {vol.shape} {vol.dtype}  slice {k} identical')

## 4. Looking at the data

In [ ]:
cols = ['n_slices', 'rows', 'cols']
print(idx[cols].describe().loc[['min', '50%', 'max']].T)
print(idx.pixel_dtype.value_counts().to_dict(), '| size on disk (GB):', round(sum(os.path.getsize(ROOT / p) for p in idx.array_path) / 1e9, 2))
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
idx.n_slices.clip(upper=150).plot.hist(bins=40, ax=ax[0], title='Slices per series (clipped at 150)')
idx.groupby(['rows', 'cols']).size().sort_values(ascending=False).head(10).plot.bar(ax=ax[1], title='Top in-plane sizes (rows, cols)', rot=45)
idx.groupby('Anatomical_Plane').n_slices.median().plot.bar(ax=ax[2], title='Median slices by plane', rot=0)
plt.tight_layout(); plt.show()

In [ ]:
# Which tags exist and how variable are they?
summary = pd.DataFrame({'non_null': meta.notna().mean().round(3), 'n_unique': meta.nunique()})
summary.sort_values('n_unique')

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(15, 8))
for a, c in zip(ax.ravel(), ['Manufacturer', 'MagneticFieldStrength', 'MRAcquisitionType', 'ScanningSequence', 'PatientSex', 'TransferSyntax']):
    meta.drop_duplicates('SeriesInstanceUID')[c].value_counts(dropna=False).head(10).plot.barh(ax=a, title=c)
plt.tight_layout(); plt.show()

In [ ]:
s = meta.drop_duplicates('SeriesInstanceUID')
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
s.PixelSpacing.map(lambda v: json.loads(v)[0] if isinstance(v, str) else np.nan).plot.hist(bins=40, ax=ax[0], title='In-plane pixel spacing (mm)')
s.SliceThickness.plot.hist(bins=40, ax=ax[1], title='Slice thickness (mm)')
s.RepetitionTime.clip(upper=8000).plot.hist(bins=40, ax=ax[2], title='TR (ms, clipped)')
plt.tight_layout(); plt.show()

### One study: every series, middle slice

In [ ]:
def show_study(study_uid, ncols=6):
    ss = idx[idx.StudyInstanceUID == study_uid].sort_values(['Anatomical_Plane', 'SeriesInstanceUID'])
    n = len(ss); nrows = int(np.ceil(n / ncols))
    fig, ax = plt.subplots(nrows, ncols, figsize=(3 * ncols, 3.2 * nrows), squeeze=False)
    for a in ax.ravel(): a.axis('off')
    for a, (_, r) in zip(ax.ravel(), ss.iterrows()):
        vol = np.load(ROOT / r.array_path)
        a.imshow(vol[len(vol) // 2], cmap='gray', vmin=np.percentile(vol, 1), vmax=np.percentile(vol, 99.5))
        a.set_title(f'{r.Anatomical_Plane}  fluid={r.Fluid_Sensitive} fat={r.Fat_Suppression}\n{vol.shape} {vol.dtype}', fontsize=8)
    plt.suptitle(study_uid[-12:]); plt.tight_layout(); plt.show()

labeled_ids = set(lab.StudyInstanceUID)
counts = idx.StudyInstanceUID.value_counts()          # prefer a labeled study with the most processed series
lab_counts = counts[counts.index.isin(labeled_ids)]
demo = (lab_counts if len(lab_counts) else counts).index[0]
print(train.set_index('StudyInstanceUID').loc[demo, LABELS].to_dict())
print(train.set_index('StudyInstanceUID').loc[demo, 'Report'][:600])
show_study(demo)

### A series as a montage of slices

In [ ]:
r = idx[(idx.Anatomical_Plane == 'Sagittal') & (idx.Fluid_Sensitive == 1)].iloc[0]
vol = np.load(ROOT / r.array_path)
ks = np.linspace(0, len(vol) - 1, 12).astype(int)
fig, ax = plt.subplots(2, 6, figsize=(18, 6.4))
for a, k in zip(ax.ravel(), ks):
    a.imshow(vol[k], cmap='gray', vmin=np.percentile(vol, 1), vmax=np.percentile(vol, 99.5)); a.set_title(f'slice {k}'); a.axis('off')
plt.suptitle(f'Sagittal fluid-sensitive series {vol.shape} {vol.dtype}'); plt.tight_layout(); plt.show()

### Intensity ranges are not comparable across series (hence the raw values + metadata)

In [ ]:
ex = idx.sample(min(300, len(idx)), random_state=SEED)
st = []
for p in ex.array_path:
    v = np.load(ROOT / p, mmap_mode='r')
    st.append((v.min(), np.percentile(v[len(v)//2], 99), v.max()))
st = pd.DataFrame(st, columns=['min', 'p99_mid_slice', 'max'], index=ex.index)
st['pixel_dtype'] = ex.pixel_dtype
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
st.p99_mid_slice.plot.hist(bins=50, ax=ax[0], title='99th pct of middle slice'); ax[0].set_yscale('log')
st.groupby('pixel_dtype').p99_mid_slice.median().plot.bar(ax=ax[1], title='median by dtype', rot=0)
plt.tight_layout(); plt.show()
st.describe()